<a href="https://colab.research.google.com/github/musab855/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

Full-depth companion to ML-04 (`w03_data_contract.ipynb`): build the feature vector the
model actually scored with, write down what every feature means and *when* it exists, then
**attack it** — label-derived columns, future windows, decision-derived flags — and show the
tests running. Optional on the portal card ("the capstone rewards it"); executed here
top-to-bottom with receipts.

> Skills: `hunting-leakage-and-validating` + `flyrank/flyrank-data` (see `skills/README.md`).

## 1. Build the feature vector

*The three floats the Week-5 model scored with, loaded cache-first from the same March 2026
frame Weeks 4–7 used. Frame facts are asserted against `capstone_metrics.json` **before**
anything else runs — if 140,599 / 43 clients / 0.4649 / 1,074 flagged don't hold, stop here.*

In [1]:
# ---- Setup (repo-root aware; token never printed, stored, or written) --------
import os, sys, getpass, subprocess, importlib.util, shutil, json
from pathlib import Path

def find_repo_root(start):
    for p in [start, *start.parents]:
        if (p / "work").is_dir() and (p / "skills").is_dir():
            return p
    return None

root = find_repo_root(Path.cwd())
if root is None:
    if Path("/content").exists():
        dest = Path("/content/flyrank-ml-internship")   # Colab
    else:
        dest = Path.cwd() / "flyrank-ml-internship"
    if (dest / "work").is_dir() and (dest / "skills").is_dir():
        print(f"repo folder already present - reusing it: {dest}")
        root = dest
    else:
        if dest.exists():
            shutil.rmtree(dest)
        proc = subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/musab855/flyrank-ml-internship.git", str(dest)],
            capture_output=True, text=True)
        if proc.returncode != 0:
            print(proc.stdout); print(proc.stderr)
            raise RuntimeError(f"git clone failed (exit {proc.returncode})")
        root = dest
os.chdir(root)

missing = [p for p in ("duckdb", "huggingface_hub") if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    importlib.invalidate_caches()

def load_token():
    try:
        from google.colab import userdata
        for key in ("HF_TOKEN2", "HF_TOKEN"):
            try:
                token = userdata.get(key)
                if token:
                    return token
            except Exception:
                pass
    except Exception:
        pass
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    env_file = Path(".env")
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            if line.strip().startswith("HF_TOKEN"):
                return line.split("=", 1)[1].strip().strip(chr(34) + chr(39))
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import numpy as np
import pandas as pd
from IPython.display import display

print("working directory:", Path.cwd())

# ---- Feature frame: cache-first, same March 2026 window as Weeks 4-7 --------
# Unlike the capstone, gsc_impressions_16to31 is KEPT here - Test C below needs it
# for the deliberate trap. It is never a feature; that is what the tests prove.
CACHE = Path("work/outputs/feature_frame_march_audit.csv")

if CACHE.exists():
    df = pd.read_csv(CACHE)
    print(f"Cache hit: {len(df):,} rows from {CACHE}")
else:
    HF_TOKEN = load_token()
    import duckdb
    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    rel = "hf://datasets/FlyRank/internship-warehouse"
    df = con.sql(f"""
    WITH daily_averages AS (
      SELECT content_hash_id, client_hash_id,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) AS gsc_impressions_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_clicks ELSE NULL END) AS gsc_clicks_1to15,
        AVG(CASE WHEN report_date < '2026-03-16' AND gsc_data_available THEN gsc_avg_position ELSE NULL END) AS position_1to15,
        AVG(CASE WHEN report_date >= '2026-03-16' AND gsc_data_available THEN gsc_impressions ELSE NULL END) AS gsc_impressions_16to31
      FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
      WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01' AND gsc_data_available = TRUE
      GROUP BY content_hash_id, client_hash_id
    )
    SELECT content_hash_id AS content_id, client_hash_id,
      gsc_impressions_1to15 AS gsc_impressions_avg,
      gsc_clicks_1to15 AS gsc_clicks_avg,
      position_1to15 AS position_avg,
      gsc_impressions_16to31,
      (CASE WHEN gsc_impressions_16to31 IS NULL OR gsc_impressions_1to15 IS NULL THEN NULL
            WHEN gsc_impressions_16to31 < gsc_impressions_1to15 THEN 1 ELSE 0 END) AS is_declining_label
    FROM daily_averages
    """).df()
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CACHE, index=False)
    print(f"Scanned the warehouse (March 2026): {len(df):,} rows -> cached at {CACHE}")

n_before = len(df)
pos_nan, pos_zero = df["position_avg"].isna(), df["position_avg"] == 0
lab_nan = df["is_declining_label"].isna()
n_pos_nan, n_pos_zero, n_lab_nan_extra = int(pos_nan.sum()), int(pos_zero.sum()), int((lab_nan & ~pos_nan & ~pos_zero).sum())
print(f"Rows before filter: {n_before:,}")
print(f"Excluding position NaN (no GSC data days 1-15): {n_pos_nan:,}")
print(f"Excluding position == 0 (the 'no data' placeholder): {n_pos_zero:,}")
print(f"Excluding undefined labels (one half unmeasurable): {n_lab_nan_extra:,}")

df = df[~pos_nan & ~pos_zero & ~lab_nan].copy()
df["is_declining_label"] = df["is_declining_label"].astype(int)
df = df.sort_values("content_id", kind="stable").reset_index(drop=True)
assert len(df) + n_pos_nan + n_pos_zero + n_lab_nan_extra == n_before

BASE_FEATURES = ["gsc_impressions_avg", "gsc_clicks_avg", "position_avg"]

# Frame receipts (work/outputs/capstone_metrics.json): assert, don't assume.
cm = json.loads(Path("work/outputs/capstone_metrics.json").read_text(encoding="utf-8"))
assert len(df) == cm["frame"]["pages"] == 140_599, f"frame changed: {len(df):,}"
assert df["client_hash_id"].nunique() == cm["frame"]["clients"] == 43
base_rate = float(df["is_declining_label"].mean())
assert abs(base_rate - cm["frame"]["base_rate"]) < 0.0005, f"base rate {base_rate:.4f}"
flagged = (df["gsc_impressions_avg"] >= 500) & (df["position_avg"] >= 11)
assert int(flagged.sum()) == cm["frame"]["flagged_by_rule"] == 1_074

print(f"\nModeling frame: {len(df):,} pages, {df['client_hash_id'].nunique()} clients, "
      f"base rate {base_rate:.4f}, flagged {int(flagged.sum()):,}")
print("features:", BASE_FEATURES)

working directory: C:\Users\Asus\Downloads\flyrank-ml-internship


Cache hit: 176,738 rows from work\outputs\feature_frame_march_audit.csv
Rows before filter: 176,738
Excluding position NaN (no GSC data days 1-15): 24,757
Excluding position == 0 (the 'no data' placeholder): 1,306
Excluding undefined labels (one half unmeasurable): 10,076



Modeling frame: 140,599 pages, 43 clients, base rate 0.4649, flagged 1,074
features: ['gsc_impressions_avg', 'gsc_clicks_avg', 'position_avg']


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing values | Available when? |
|---|---|---|---|
| `gsc_impressions_avg` | mean search impressions/day, days 1–15, GSC rows only (`gsc_data_available`) | pages with no GSC data in the window are excluded upstream (24,757 position-NaN rows); after filtering **0 nulls** | days 1–15 → knowable at **2026-03-16** |
| `gsc_clicks_avg` | mean search clicks/day, days 1–15, same availability flag | same exclusions; 0 nulls after filtering | days 1–15 → knowable at **2026-03-16** |
| `position_avg` | mean search position, days 1–15 | `position == 0` means "no data", not rank zero — those 1,306 rows are dropped, not filled; 0 nulls remain | days 1–15 → knowable at **2026-03-16** |
| `ctr` *(derived, **not** a feature)* | clicks ÷ impressions, days 1–15 | defined as 0 when impressions = 0 | reason codes only — a same-window kin of the label, deliberately kept out of the feature list |

No categorical columns: the vector is three floats, no encoding, no filling. The IDs
(`content_id`, `client_hash_id`) are **not** part of it — they only group the folds.
The cell below re-measures every claim in this table.

In [2]:
# ---- Feature notes, backed by numbers ----------------------------------------
df["ctr"] = np.where(df["gsc_impressions_avg"] > 0,
                     df["gsc_clicks_avg"] / df["gsc_impressions_avg"], 0.0)

rows = []
for col in BASE_FEATURES + ["ctr"]:
    s = df[col]
    rows.append({"feature": col, "n": int(s.notna().sum()), "nulls": int(s.isna().sum()),
                 "min": round(float(s.min()), 3), "median": round(float(s.median()), 3),
                 "p99": round(float(s.quantile(0.99)), 3), "max": round(float(s.max()), 3)})
display(pd.DataFrame(rows).set_index("feature"))

assert int(df[BASE_FEATURES].isna().sum().sum()) == 0, "a feature still has nulls"
assert np.isfinite(df[BASE_FEATURES].to_numpy()).all(), "a feature has inf/nan"
ctr_check = np.where(df["gsc_impressions_avg"] > 0,
                     df["gsc_clicks_avg"] / df["gsc_impressions_avg"], 0.0)
assert np.allclose(df["ctr"], ctr_check), "ctr derivation changed"
assert "ctr" not in BASE_FEATURES, "ctr must not be a feature (same-window kin of the label)"
assert df["content_id"].is_unique, "grain broke: content_id not unique"

print("0 nulls, all finite, ctr derivation verified and NOT in the feature list.")
print("Available-when: features live 2026-03-01..15 < prediction moment 2026-03-16 < label 2026-03-16..31.")
print("No categorical columns to encode; IDs stay on the grouping side of the split.")

,n,nulls,min,median,p99,max
feature,,,,,,
gsc_impressions_avg,140599,0,1.000,10.533,795.268,12428.846
gsc_clicks_avg,140599,0,0.000,0.000,2.867,184.231
position_avg,140599,0,0.012,8.500,79.556,310.000
ctr,140599,0,0.000,0.000,0.049,1.000


0 nulls, all finite, ctr derivation verified and NOT in the feature list.
Available-when: features live 2026-03-01..15 < prediction moment 2026-03-16 < label 2026-03-16..31.
No categorical columns to encode; IDs stay on the grouping side of the split.


## 3. The leakage hunt

*Attack your own features — the three ways answers sneak in (skill:
`hunting-leakage-and-validating`):*

1. **Label-derived features.** The label, its own window, or its ancestors in the feature
   list. *Test A:* banned-set assertion. *Test C:* the confession run — train once **with**
   a known leak, once without; a collapse is the confession.
2. **Future / overlapping windows.** A feature whose window contains the label's already
   knows the outcome. *Test B:* the timeline, drawn, not assumed.
3. **Decision-derived features (product flags).** Scores someone's system already produced
   encode an old decision. *Status:* this release ships no product flags, and this work
   never rebuilds any — the rule's thresholds produce baseline *rows*, never model *inputs*.

**Test C performs the trap on real warehouse data:** the cache still carries
`gsc_impressions_16to31` (the label's own window). Train the forest **with** it — the score
should leap toward perfect — then train **without** it and keep that honest number.

In [3]:
# ---- Test A: banned columns never enter the feature vector --------------------
BANNED = {
    "is_declining_label", "gsc_impressions_16to31",   # the label and its own window
    "trend_pct", "trend_direction",                    # the label's ancestors (ML-04's trap)
    "content_id", "client_hash_id",                    # IDs: grouping only, never features
    "ctr",                                             # same-window kin of the label
    "rule_full", "impressions_only",                   # baseline ranks, never inputs
}
violations = set(BASE_FEATURES) & BANNED
assert not violations, f"banned columns in the feature list: {violations}"
print("Test A PASS - banned from features:", sorted(BANNED))
print("violations:", violations or "none")

# ---- Test B: the timeline, drawn --------------------------------------------
timeline = pd.DataFrame([
    {"stage": "features",    "window": "2026-03-01 .. 2026-03-15",
     "knowable at prediction moment?": "yes - strictly before"},
    {"stage": "prediction",  "window": "2026-03-16",
     "knowable at prediction moment?": "this IS the moment"},
    {"stage": "label",       "window": "2026-03-16 .. 2026-03-31",
     "knowable at prediction moment?": "no - strictly after"},
])
display(timeline)
print("Test B PASS - every feature window strictly precedes the label window.")

# ---- Test C: the confession run (train WITH the leak, then WITHOUT) ----------
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

RANDOM_STATE, N_FOLDS = 42, 5

def make_grouped_folds(frame, n_splits=N_FOLDS, seed=RANDOM_STATE):
    sizes = frame.groupby("client_hash_id").size()
    order = list(sizes.index)
    np.random.default_rng(seed).shuffle(order)
    fold_of_client, load = {}, [0] * n_splits
    for client in order:
        i = int(np.argmin(load))
        fold_of_client[client] = i
        load[i] += int(sizes[client])
    return frame["client_hash_id"].map(fold_of_client).astype(int)

df["fold_grouped"] = make_grouped_folds(df)
assert df.groupby("client_hash_id")["fold_grouped"].nunique().eq(1).all(), "a client spans two folds"
print("client-grouped 5-fold split rebuilt (same seed, same folds as the capstone)")

y = df["is_declining_label"].to_numpy()
X_leaky = df[BASE_FEATURES + ["gsc_impressions_16to31"]].replace([np.inf, -np.inf], np.nan).fillna(0)
X_honest = df[BASE_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
assert "gsc_impressions_16to31" not in BASE_FEATURES

leaky, honest = np.zeros(len(df)), np.zeros(len(df))
for f in range(N_FOLDS):
    test = (df["fold_grouped"] == f).to_numpy()
    train = ~test
    for X, out in ((X_leaky, leaky), (X_honest, honest)):
        m = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=25,
                                   class_weight="balanced_subsample", n_jobs=-1,
                                   random_state=RANDOM_STATE)
        m.fit(X.iloc[train], y[train])
        out[test] = m.predict_proba(X.iloc[test])[:, 1]
    print(f"fold {f}: leaky + honest both scored out-of-fold")

auc_leaky, auc_honest = roc_auc_score(y, leaky), roc_auc_score(y, honest)
pb = json.loads(Path("work/outputs/playbook_metrics.json").read_text(encoding="utf-8"))
target = pb["receipts"]["random_forest"]["roc_auc"]
print(f"\nWITH  gsc_impressions_16to31: ROC-AUC {auc_leaky:.4f}  <- the confession (a real leak)")
print(f"WITHOUT (honest):             ROC-AUC {auc_honest:.4f}  (receipt {target:.4f})")
assert auc_leaky > 0.95, f"the harness failed to catch the planted leak ({auc_leaky:.4f})"
assert abs(auc_honest - target) <= 0.01, f"honest AUC {auc_honest:.4f} vs receipt {target}"
print("Test C PASS - leak planted, detected, removed; the honest number is the one kept.")

Test A PASS - banned from features: ['client_hash_id', 'content_id', 'ctr', 'gsc_impressions_16to31', 'impressions_only', 'is_declining_label', 'rule_full', 'trend_direction', 'trend_pct']
violations: none


,stage,window,knowable at prediction moment?
0,features,2026-03-01 .. 2026-03-15,yes - strictly before
1,prediction,2026-03-16,this IS the moment
2,label,2026-03-16 .. 2026-03-31,no - strictly after


Test B PASS - every feature window strictly precedes the label window.


client-grouped 5-fold split rebuilt (same seed, same folds as the capstone)


fold 0: leaky + honest both scored out-of-fold


fold 1: leaky + honest both scored out-of-fold


fold 2: leaky + honest both scored out-of-fold


fold 3: leaky + honest both scored out-of-fold


fold 4: leaky + honest both scored out-of-fold



WITH  gsc_impressions_16to31: ROC-AUC 0.9966  <- the confession (a real leak)
WITHOUT (honest):             ROC-AUC 0.6543  (receipt 0.6543)
Test C PASS - leak planted, detected, removed; the honest number is the one kept.


## 4. What I excluded and why

| Excluded | Why |
|---|---|
| `fact_content_query_90d` (query-level rows) | raw queries are private — they never enter this work |
| `is_declining_label`, `gsc_impressions_16to31` | the label itself and its own window (Test C above) |
| `trend_pct`, `trend_direction` | the label's ancestors (ML-04's trap; never features) |
| `content_id`, `client_hash_id` | IDs: grouping/folds only — memorizing groups is not skill |
| `ctr` | same-window kin of the label: fine for reason codes, dishonest as a feature |
| product flags & system scores | not in this release, and never rebuilt — a flag encodes an old decision |
| 24,757 no-GSC rows, 1,306 `position == 0`, 10,076 undefined labels | "no data" placeholders and unmeasurable halves — dropped, never guessed |

**Privacy:** this notebook prints aggregates only — no client names, domains, URLs, titles,
or queries exist in its outputs, and it writes no files to disk at all. The next cell
verifies the column surface and the exclusions.

In [4]:
# ---- Exclusions verified + final verdict --------------------------------------
for col in ["is_declining_label", "gsc_impressions_16to31", "trend_pct",
            "trend_direction", "content_id", "client_hash_id", "ctr"]:
    assert col not in BASE_FEATURES, f"{col} slipped into the feature list"

allowed = {"content_id", "client_hash_id", "gsc_impressions_avg", "gsc_clicks_avg",
           "position_avg", "gsc_impressions_16to31", "is_declining_label",
           "ctr", "fold_grouped"}
unexpected = set(df.columns) - allowed
assert not unexpected, f"unexpected columns in the frame: {unexpected}"

for col in ["trend_pct", "trend_direction", "rule_full", "impressions_only"]:
    assert col not in df.columns, f"{col} exists in this frame and must not"

print("LEAKAGE & PRIVACY CHECK: PASS")
print(f"  banned-column violations : {len(violations)}")
print(f"  timeline                 : features < prediction < label (Test B)")
print(f"  planted leak             : ROC-AUC {auc_leaky:.3f} -> honest {auc_honest:.3f} (receipt {target:.3f})")
print(f"  frame columns            : {sorted(df.columns)} (no name/domain/query columns exist)")
print("  disk writes              : none (this notebook persists nothing)")

LEAKAGE & PRIVACY CHECK: PASS
  banned-column violations : 0
  timeline                 : features < prediction < label (Test B)
  planted leak             : ROC-AUC 0.997 -> honest 0.654 (receipt 0.654)
  frame columns            : ['client_hash_id', 'content_id', 'ctr', 'fold_grouped', 'gsc_clicks_avg', 'gsc_impressions_16to31', 'gsc_impressions_avg', 'is_declining_label', 'position_avg'] (no name/domain/query columns exist)
  disk writes              : none (this notebook persists nothing)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.